## Unidad 15 · NLP en entornos productivos

En este ejercicio trabajaremos con el dataset **Customer Support on Twitter (twcs.csv)** para construir un pipeline automático de NLP.  
El objetivo es simular un caso real de negocio donde una empresa recibe miles de consultas diarias y necesita:

- Procesar grandes volúmenes de texto.
- Implementar pipelines automáticos de limpieza y análisis.
- Aplicar análisis de sentimiento y clasificación simple.
- Calcular métricas relevantes para soporte al cliente.
- Reflexionar sobre escalabilidad y ROI.


In [5]:
# TextBlob no siempre viene instalado por defecto
!pip install textblob

# -------------------------
# 2. Importación de librerías
# -------------------------
import pandas as pd
import numpy as np
import re
import os

from textblob import TextBlob
from google.colab import files

import kagglehub

# -------------------------
# 3. Subida del dataset desde kaggle
# -------------------------

path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")

print("Path to dataset files:", path)

print("Contents of downloaded directory:", os.listdir(path))

# -------------------------
# 4. Carga del CSV en un DataFrame
# -------------------------

file_name = os.path.join(path, 'twcs', 'twcs.csv')

df = pd.read_csv(file_name)

# -------------------------
# 5. Inspección inicial del dataset
# -------------------------
print("Dimensiones del dataset:")
print(df.shape)

print("\nColumnas disponibles:")
print(df.columns)

print("\nPrimeras filas:")
df.head()

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Path to dataset files: /kaggle/input/customer-support-on-twitter
Contents of downloaded directory: ['twcs', 'sample.csv']
Dimensiones del dataset:
(2811774, 7)

Columnas disponibles:
Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

Primeras filas:


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


### Paso 6 – Limpieza de texto
En este paso normalizamos los tweets para que el análisis sea más confiable:
- Convertimos a minúsculas.
- Eliminamos URLs y menciones.
- Quitamos caracteres especiales.

Esto permite que el modelo procese texto homogéneo y reduzca ruido.


In [6]:
# -------------------------
# 6. Limpieza de texto
# -------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # URLs
    text = re.sub(r'@\w+', '', text)  # menciones
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # caracteres especiales
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)
print("\nEjemplo de limpieza:")
df[['text','clean_text']].head()



Ejemplo de limpieza:


,text,clean_text
0,@115712 I understand. I would like to assist y...,i understand i would like to assist you we wou...
1,@sprintcare and how do you propose we do that,and how do you propose we do that
2,@sprintcare I have sent several private messag...,i have sent several private messages and no on...
3,@115712 Please send us a Private Message so th...,please send us a private message so that we ca...
4,@sprintcare I did.,i did


### Paso 7 – Análisis de sentimiento
Aplicamos **TextBlob** para obtener la polaridad de cada tweet:
- Valores negativos indican insatisfacción.
- Valores positivos reflejan satisfacción.
- Valores cercanos a cero se consideran neutros.

Esto nos ayuda a medir la percepción de los clientes en tiempo real.


In [7]:
# -------------------------
# 7. Análisis de sentimiento
# -------------------------
def get_sentiment(text):
    blob = TextBlob(text)
    return blob.sentiment.polarity

df['sentiment'] = df['clean_text'].apply(get_sentiment)

# Clasificación simple del sentimiento
df['sentiment_label'] = df['sentiment'].apply(
    lambda x: 'Negativo' if x < -0.1 else ('Positivo' if x > 0.1 else 'Neutro')
)

print("\nDistribución de sentimiento:")
print(df['sentiment_label'].value_counts(normalize=True) * 100)



Distribución de sentimiento:
sentiment_label
Neutro      51.48056
Positivo    31.14724
Negativo    17.37220
Name: proportion, dtype: float64


### Paso 8 – Clasificación de consultas
Definimos categorías simples:
- **Facturación**: menciones a cobros, facturas.
- **Soporte Técnico**: problemas, errores.
- **Reclamos**: cualquier otra consulta.

Esto permite segmentar automáticamente los tickets y priorizar los críticos.


In [8]:
# -------------------------
# 8. Clasificación de consultas
# -------------------------
def classify_query(text):
    if 'bill' in text or 'charge' in text or 'factura' in text:
        return 'Facturación'
    elif 'error' in text or 'problem' in text or 'soporte' in text:
        return 'Soporte Técnico'
    else:
        return 'Reclamo'

df['category'] = df['clean_text'].apply(classify_query)

print("\nDistribución de categorías:")
print(df['category'].value_counts(normalize=True) * 100)



Distribución de categorías:
category
Reclamo            95.267329
Soporte Técnico     2.697550
Facturación         2.035121
Name: proportion, dtype: float64


### Paso 9 – Métricas de negocio
Calculamos indicadores clave:
- **Tasa de automatización**: % de consultas clasificadas automáticamente.
- **Distribución de sentimiento**: proporción de tweets negativos, neutros y positivos.
- **Tiempo de respuesta promedio (TTR)**: simulamos reducción de 10 min manual a 2 min automático.

Estas métricas permiten justificar el ROI y mostrar impacto tangible.


In [9]:
# -------------------------
# 9. Métricas de negocio
# -------------------------

# Tasa de automatización (ejemplo: % de tweets clasificados automáticamente)
automation_rate = df['category'].notnull().mean() * 100

# Distribución de sentimiento
sentiment_dist = df['sentiment_label'].value_counts(normalize=True) * 100

print("Tasa de automatización:", automation_rate, "%")
print("\nDistribución de sentimiento:\n", sentiment_dist)

# Simulación de TTR: supongamos 10 min manual vs 2 min automático
manual_TTR = 10
auto_TTR = 2
avg_TTR = (automation_rate/100)*auto_TTR + ((100-automation_rate)/100)*manual_TTR
print("\nTiempo de respuesta promedio simulado:", avg_TTR, "minutos")


Tasa de automatización: 100.0 %

Distribución de sentimiento:
 sentiment_label
Neutro      51.48056
Positivo    31.14724
Negativo    17.37220
Name: proportion, dtype: float64

Tiempo de respuesta promedio simulado: 2.0 minutos


## Paso 10 – Reflexión final: escalabilidad y ROI

Los resultados obtenidos permiten extraer conclusiones claras sobre el valor de implementar NLP en entornos productivos:

#### Escalabilidad Estratégica: Pilar para el Crecimiento Sostenible

La infraestructura de Procesamiento de Lenguaje Natural (PLN) implementada ha demostrado una robusta **escalabilidad**, siendo capaz de procesar eficientemente miles de tweets y manejar volúmenes masivos de datos sin degradación del rendimiento. Esta capacidad es crucial para la empresa, ya que:

*   **Soporta la Expansión Futura**: Permite absorber el crecimiento exponencial de las interacciones con clientes a medida que la empresa se expande, sin requerir inversiones desproporcionadas en infraestructura o personal adicional para el procesamiento de datos.
*   **Garantiza la Continuidad Operativa**: Mantiene la agilidad y la capacidad de respuesta frente a picos de demanda o eventos inesperados, asegurando que la gestión de la relación con el cliente no se vea comprometida.
*   **Optimiza la Inversión a Largo Plazo**: La solidez de la arquitectura actual minimiza los riesgos de obsolescencia tecnológica y maximiza el retorno de la inversión inicial, al garantizar que la solución seguirá siendo relevante y eficiente en un entorno de negocio dinámico.


- **Automatización**: la tasa de automatización alcanzó el 100%, eliminando la necesidad de clasificación manual en consultas simples y liberando al equipo de soporte para tareas más complejas.


#### Retorno de la Inversión (ROI): Transformando la Operación y la Experiencia del Cliente

La implementación del pipeline de PLN no solo optimiza procesos, sino que genera un **retorno de inversión (ROI)** claro y cuantificable, tanto a nivel operativo como estratégico:

*   **Ahorros Operacionales Directos:** La tasa de automatización del 100% en la clasificación de consultas simples es un pilar fundamental. Al reducir el Tiempo de Respuesta Promedio (TTR) de 10 minutos (manejo manual) a tan solo 2 minutos (manejo automatizado), logramos un ahorro del 80% en el tiempo de resolución por cada consulta. Esto se traduce directamente en una **liberación significativa de horas de personal de soporte**, permitiendo la reasignación a tareas de mayor valor añadido o una reducción en los costes operativos asociados al volumen de atención al cliente. Estos ahorros representan una mejora sustancial en la eficiencia y la rentabilidad del equipo de soporte.

*   **Valor Estratégico y Retención de Clientes:** Más allá de los ahorros directos, el sistema impacta positivamente en la experiencia del cliente. El análisis de sentimiento reveló que un **17.37% de los tweets son de naturaleza negativa**. La capacidad de identificar y priorizar automáticamente estos casos críticos permite una respuesta más rápida y efectiva, lo que es vital para mitigar el riesgo de "churn" (pérdida de clientes) y fortalecer la lealtad a la marca. Una gestión proactiva de la insatisfacción se traduce en **mayor satisfacción del cliente, una mejor reputación de marca y, en última instancia, en una mayor retención y valor de vida del cliente (CLV)**, lo que asegura ingresos recurrentes y un crecimiento sostenible a largo plazo.

#### Valor Estratégico para la Toma de Decisiones:

Los sólidos cimientos de **escalabilidad** y el cuantificable **ROI** que proporciona nuestro pipeline de PLN se traducen directamente en una **ventaja estratégica inestimable para la toma de decisiones gerenciales**. Ya no solo operamos de manera más eficiente, sino que obtenemos una inteligencia de negocio accionable que permite anticipar, reaccionar y evolucionar de forma proactiva.

*   **Mitigación Proactiva de Riesgos y Mejora de la Experiencia del Cliente:** El análisis de sentimiento es una herramienta clave en este aspecto. Saber que un **17.37% de los tweets son negativos** no es solo un dato; es una alerta temprana. Esta información permite a la gerencia identificar rápidamente tendencias de insatisfacción, puntos de fricción en la experiencia del cliente o fallos recurrentes en productos/servicios. Con estos insights, se pueden desplegar acciones correctivas de forma inmediata, evitando que pequeños problemas escalen a crisis de reputación o a una pérdida masiva de clientes. La capacidad de priorizar estos casos críticos asegura que los recursos se dirijan donde son más necesarios, mejorando la percepción de la marca y fidelizando a los clientes más valiosos.

*   **Optimización Continua de Productos y Servicios:** La clasificación automática de consultas, complementada con el análisis de sentimiento, genera un flujo constante de feedback estructurado. Al categorizar sistemáticamente las consultas (Facturación, Soporte Técnico, Reclamos), la gerencia puede identificar qué áreas generan mayor volumen de consultas o quejas. Esto ofrece una visión clara para:
    *   **Mejorar la documentación o FAQs** para reducir consultas repetitivas.
    *   **Identificar áreas de mejora en el producto o servicio** que generan problemas técnicos o de usabilidad.
    *   **Optimizar los procesos internos** de soporte al cliente, asignando recursos de manera más eficiente y diseñando soluciones a largo plazo.



### Conclusión Gerencial

El análisis detallado de la implementación de nuestro pipeline de Procesamiento de Lenguaje Natural (PLN) en la gestión de interacciones con clientes confirma que esta solución es mucho más que una mejora operativa; es un **activo estratégico fundamental** que impulsa el crecimiento y la eficiencia del negocio. Los pilares de **escalabilidad**, **retorno de inversión (ROI)** y **valor estrat\u00e9gico para la toma de decisiones** se entrelazan para ofrecer una ventaja competitiva sostenible.

Hemos demostrado que la plataforma es **intrínsecamente escalable**, capaz de manejar el volumen creciente de interacciones con clientes sin incurrir en costes desproporcionados, garantizando la continuidad operativa y protegiendo nuestra inversión a largo plazo. En términos de **ROI, la automatización del 100%** de consultas simples ha reducido drásticamente el Tiempo de Respuesta Promedio (TTR) de 10 a 2 minutos, liberando recursos valiosos del equipo de soporte y generando ahorros operativos directos. Además, la capacidad de identificar y reaccionar proactivamente ante el **17.37% de sentimiento negativo** no solo mejora la satisfacción y retención de clientes, sino que también protege nuestra reputación de marca y el Valor de Vida del Cliente (CLV).

En última instancia, este sistema nos proporciona una **inteligencia de negocio en tiempo real**, transformando el feedback del cliente en insights accionables que permiten a la gerencia tomar decisiones informadas para la mitigación de riesgos, la optimización continua de productos y servicios, y la mejora de la experiencia del cliente. La inversión en esta solución de PLN no solo está justificada por los ahorros y eficiencias generadas, sino por su capacidad para fortalecer nuestra posición en el mercado y asegurar un crecimiento empresarial sostenible y rentable..
